# Sample Telemetry Notebook
This notebook shows how placeholder tokens can inject environment-specific
values into telemetry and monitoring code.

### Placeholders used in this notebook
| Token | Replaced With |
|---|---|
| `##logAnalyticsWorkspaceId##` | Log Analytics workspace GUID |
| `##fabricWorkspaceId##` | Workspace GUID |
| `##fabricLakehouseId##` | Lakehouse GUID |
| `##environmentName##` | Environment suffix (dev / test / prod) |
| `##tenantName##` | Tenant naming prefix |

In [ ]:
# ---------------------------------------------------------------
# Configuration — ##placeholders## are swapped at deploy time
# ---------------------------------------------------------------
LOG_ANALYTICS_ID  = "##logAnalyticsWorkspaceId##"
WORKSPACE_ID      = "##fabricWorkspaceId##"
LAKEHOUSE_ID      = "##fabricLakehouseId##"
ENVIRONMENT       = "##environmentName##"
TENANT            = "##tenantName##"

In [ ]:
# ---------------------------------------------------------------
# Build telemetry record
# ---------------------------------------------------------------
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

schema = StructType([
    StructField("workspace_id", StringType()),
    StructField("lakehouse_id", StringType()),
    StructField("environment", StringType()),
    StructField("tenant", StringType()),
    StructField("event_type", StringType()),
    StructField("event_time", TimestampType()),
    StructField("details", StringType()),
])

telemetry_df = (
    spark.createDataFrame([], schema)
    .union(
        spark.sql("""
            SELECT
                '{ws_id}' AS workspace_id,
                '{lh_id}' AS lakehouse_id,
                '{env}'   AS environment,
                '{tenant}' AS tenant,
                'notebook_run' AS event_type,
                current_timestamp() AS event_time,
                'SampleTelemetry executed successfully' AS details
        """.format(
            ws_id=WORKSPACE_ID,
            lh_id=LAKEHOUSE_ID,
            env=ENVIRONMENT,
            tenant=TENANT,
        ))
    )
)

telemetry_df.show(truncate=False)

In [ ]:
# ---------------------------------------------------------------
# Write telemetry to Lakehouse delta table
# ---------------------------------------------------------------
target = f"abfss://{LAKEHOUSE_ID}@onelake.dfs.fabric.microsoft.com/Tables/telemetry_log"

telemetry_df.write.format("delta").mode("append").save(target)

print(f"Telemetry written to {target}")

In [ ]:
# ---------------------------------------------------------------
# Optional — push summary to Log Analytics
# ---------------------------------------------------------------
import json

log_payload = {
    "workspace_id": WORKSPACE_ID,
    "lakehouse_id": LAKEHOUSE_ID,
    "log_analytics_workspace": LOG_ANALYTICS_ID,
    "environment": ENVIRONMENT,
    "tenant": TENANT,
    "status": "success",
}

print("Log Analytics payload:")
print(json.dumps(log_payload, indent=2))

# In production, use the Log Analytics Data Collector API:
# POST https://{LOG_ANALYTICS_ID}.ods.opinsights.azure.com/api/logs?api-version=2016-04-01